In [149]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [150]:
DATA_path="../Bangladesh_Multi_Site_Air_Quality.csv"
TRAIN_OUT="train_clean.csv"
TEST_OUT="test_clear.csv"

In [151]:
#load ar building data time
df=pd.read_csv(DATA_path)
df.columns


Index(['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2',
       'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station'],
      dtype='str')

In [152]:
df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
df

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,datetime
0,1,2022,8,4,0,15.1,21.8,11.9,28.9,262.0,13.0,26.8,1002.5,25.2,0.0,SE,2.78,Dhaka,2022-08-04 00:00:00
1,2,2022,8,4,1,11.6,16.8,12.1,25.1,245.0,18.0,27.7,1003.2,25.0,0.0,SE,3.61,Dhaka,2022-08-04 01:00:00
2,3,2022,8,4,2,11.8,17.1,12.3,19.5,221.0,25.0,28.4,1003.4,24.8,0.0,SE,4.33,Dhaka,2022-08-04 02:00:00
3,4,2022,8,4,3,9.2,13.5,12.1,13.2,193.0,35.0,29.2,1003.5,25.0,0.0,SE,3.94,Dhaka,2022-08-04 03:00:00
4,5,2022,8,4,4,9.0,13.2,11.1,10.5,180.0,43.0,30.2,1003.9,25.3,0.1,SE,3.61,Dhaka,2022-08-04 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208363,208364,2026,7,20,19,9.7,11.1,1.3,6.5,198.0,51.0,26.3,1004.1,25.1,0.2,SSE,2.76,Barisal,2026-07-20 19:00:00
208364,208365,2026,7,20,20,9.8,11.2,1.2,7.0,171.0,48.0,25.9,1003.9,25.1,1.8,SSE,2.88,Barisal,2026-07-20 20:00:00
208365,208366,2026,7,20,21,10.0,11.2,1.2,7.4,154.0,46.0,25.4,1003.9,24.7,3.1,SE,2.77,Barisal,2026-07-20 21:00:00
208366,208367,2026,7,20,22,9.6,10.8,1.2,7.5,161.0,45.0,25.4,1003.3,24.5,3.7,SSE,1.30,Barisal,2026-07-20 22:00:00


In [153]:
df = df.sort_values(["station", "datetime"]).reset_index(drop=True)
df

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,datetime
0,173641,2022,8,4,0,7.4,11.6,0.4,2.0,110.0,28.0,26.3,1003.8,25.0,0.0,SE,3.68,Barisal,2022-08-04 00:00:00
1,173642,2022,8,4,1,5.0,8.1,0.4,2.0,108.0,28.0,26.8,1004.8,24.8,0.1,SE,3.70,Barisal,2022-08-04 01:00:00
2,173643,2022,8,4,2,4.7,7.5,0.4,2.1,105.0,28.0,28.2,1004.5,25.0,0.0,SE,4.56,Barisal,2022-08-04 02:00:00
3,173644,2022,8,4,3,4.9,7.7,0.4,2.1,101.0,29.0,28.9,1004.8,25.1,0.0,SE,4.20,Barisal,2022-08-04 03:00:00
4,173645,2022,8,4,4,5.2,8.1,0.4,1.7,99.0,32.0,30.1,1004.6,25.4,0.2,SE,4.18,Barisal,2022-08-04 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208363,104180,2026,7,20,19,15.5,16.1,4.7,13.8,201.0,38.0,24.3,1003.7,23.9,2.2,E,2.18,Sylhet,2026-07-20 19:00:00
208364,104181,2026,7,20,20,14.2,14.7,3.9,12.6,206.0,40.0,24.1,1003.2,23.9,1.5,E,1.86,Sylhet,2026-07-20 20:00:00
208365,104182,2026,7,20,21,12.9,13.4,3.3,11.6,207.0,41.0,24.0,1003.1,23.7,0.9,NE,2.01,Sylhet,2026-07-20 21:00:00
208366,104183,2026,7,20,22,11.8,12.3,2.9,11.0,197.0,41.0,24.0,1003.1,23.7,1.6,ENE,2.24,Sylhet,2026-07-20 22:00:00


In [154]:
#Checking missing hourly timestamps per station
print("Checking for timestamp gaps per station...")
for station, g in df.groupby("station"):
    full_range = pd.date_range(g["datetime"].min(), g["datetime"].max(), freq="h")

    missing = full_range.difference(g["datetime"])
    
    print(f"  {station}: {len(missing)} missing hourly timestamps out of {len(full_range)}")

Checking for timestamp gaps per station...
  Barisal: 0 missing hourly timestamps out of 34728
  Chittagong: 0 missing hourly timestamps out of 34728
  Dhaka: 0 missing hourly timestamps out of 34728
  Khulna: 0 missing hourly timestamps out of 34728
  Rajshahi: 0 missing hourly timestamps out of 34728
  Sylhet: 0 missing hourly timestamps out of 34728


In [155]:
dupes = df.duplicated(subset=["station", "datetime"]).sum()
print(f"\nDuplicate (station, datetime) rows: {dupes}")
df = df.drop_duplicates(subset=["station", "datetime"])


Duplicate (station, datetime) rows: 0


In [156]:
# Columns where we want to handle extreme values
cap_cols = ["PM10", "SO2", "NO2", "CO", "O3", "WSPM"]


# This function processes one station at a time
def winsorize_group(g):

    # Process each column
    for col in cap_cols:

        # Find the 1st percentile
        lower_limit = g[col].quantile(0.01)
        

        # Find the 99th percentile
        upper_limit = g[col].quantile(0.99)


        # Check every value in this column
        for index, value in g[col].items():

            # Value is too small
            if value < lower_limit:
                g.loc[index, col] = lower_limit

            # Value is too large
            elif value > upper_limit:
                g.loc[index, col] = upper_limit

    return g


# Apply the function separately to every station
df = df.groupby("station", group_keys=False).apply(winsorize_group)


In [157]:
non_negative_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "WSPM", "RAIN"]
for col in non_negative_cols:
    n_neg = (df[col] < 0).sum()
    if n_neg > 0:
        df[col] = df[col].clip(lower=0)

In [158]:
#Time based train split test
SPLIT_DATE = "2026-01-01"
train = df[df["datetime"] < SPLIT_DATE].copy()
test = df[df["datetime"] >= SPLIT_DATE].copy()

In [159]:
print(f"\nTrain: {train.shape[0]} rows ({train['datetime'].min()} -> {train['datetime'].max()})")



Train: 179424 rows (2022-08-04 00:00:00 -> 2025-12-31 23:00:00)


In [160]:

print(f"Test:  {test.shape[0]} rows ({test['datetime'].min()} -> {test['datetime'].max()})")

Test:  28944 rows (2026-01-01 00:00:00 -> 2026-07-20 23:00:00)


In [161]:
#train.to_csv(TRAIN_OUT, index=False)
#test.to_csv(TEST_OUT, index=False)
print(f"\nSaved {TRAIN_OUT} and {TEST_OUT}")


Saved train_clean.csv and test_clear.csv
